# qb 研究侧：Notebook 原语、Overview 行与导出

对应文档：`docs/jupyter-run-detail.md`；完整 API 见 `docs/api-reference.md`。将下面 `PROJECT_ROOT` 改成你的仓库根目录（含 `app/` 与 `data/`）。

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

PROJECT_ROOT = Path(r"D:\1Codeprojects\1Backtest").resolve()
BACKEND = PROJECT_ROOT / "app" / "backend"
if str(BACKEND) not in sys.path:
    sys.path.insert(0, str(BACKEND))

RUN_ID = "demo"  # 或 "quant1-demo"

: 

In [ ]:
from app.notebook_primitives import (
    load_run_context,
    render_detail_bundle,
    show_trades_table,
)
from app.run_detail_core import load_series_response
from run_detail_research_export import (
    OverviewRow,
    build_overview_rows,
    export_run_bundle_for_detail,
    plot_overview_three_panel_plotly,
)

ctx = load_run_context(RUN_ID)
print("strategy_name:", ctx.get("strategy_name"))
bundle = render_detail_bundle(RUN_ID)
print("available_series:", bundle.available_series[:8], "...")
show_trades_table(RUN_ID).head(3)

In [ ]:
# OverviewRow 类型（TypedDict）的示例 dict
example: OverviewRow = {
    "date": "2026-01-02",
    "strategy_return": 0.1,
    "benchmark_return": 0.01,
    "excess_daily": 0.09,
    "turnover": 0.05,
    "daily_buy": 1e6,
    "daily_sell": 8e5,
}
example

In [ ]:
# 与 Web「收益概述」行数据一致：五组序列 -> build_overview_rows
eq = load_series_response(RUN_ID, "equity", "overall")["points"]
bm = load_series_response(RUN_ID, "benchmark_return", "overall")["points"]
turn = load_series_response(RUN_ID, "turnover", "overall")["points"]
db = load_series_response(RUN_ID, "daily_buy", "overall")["points"]
ds = load_series_response(RUN_ID, "daily_sell", "overall")["points"]

rows = build_overview_rows(eq, bm, turn, db, ds)
len(rows), rows[:2]

In [ ]:
# 可选：三行子图（近似 Web 三联布局）
# plot_overview_three_panel_plotly(rows).show()

In [ ]:
import pandas as pd

# 将内存中的 portfolio 等写入 artifact 目录，供浏览器 /runs/{run_id} 打开
NEW_ID = "research-export-demo"
portfolio = pd.DataFrame(
    {
        "date": pd.date_range("2026-01-01", periods=5, freq="D").strftime("%Y-%m-%d"),
        "equity": [1.0, 1.01, 1.02, 1.01, 1.03],
    }
)
manifest = {
    "run_id": NEW_ID,
    "strategy_name": "notebook-export",
    "status": "completed",
    "started_at": "2026-01-01T00:00:00Z",
    "metrics": {"total_return": 0.03},
}
export_run_bundle_for_detail(NEW_ID, manifest, portfolio, overwrite=True)
print("打开: http://127.0.0.1:5173/runs/" + NEW_ID)